# EECS E6893 HW5 — Q3 Curated Label Recheck (English)

This notebook runs **only Q3**, using the fixed 779-document CPython 3.13 dataset and the student-confirmed AI-assisted label corrections. It does not require the Northbeam starter data, an API key, or a GPU.

**Important:** The new 38/44 strict score is a **label correction**, not a model improvement. It should be described as a development-set result. Do not rebuild the curated source corpus. Use **Runtime → Change runtime type → CPU (None)**.

## 1. Upload the updated working package

Upload `HW5_Q3_Curated_Revision_20261118.zip`. It already contains `hw5_core.py`, `production.py`, the frozen Q3 docs, original and reviewed labels, and experiment scripts. Do not upload the instructor data for this Q3-only check.

In [1]:
from pathlib import Path
from google.colab import files
import zipfile

uploaded = files.upload()
archive = Path('/content/HW5_Q3_Curated_Revision_20261118.zip')
assert archive.exists(), f"Missing {archive.name}; you uploaded: {list(uploaded)}"
root = Path('/content/HW5_Q3_Recheck')
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(root)
project = root/'HW5_Official_Integrated_Code'
assert (project/'q3'/'run.py').exists(), 'Wrong ZIP layout, expected HW5_Official_Integrated_Code/q3/run.py'
assert (project/'q3'/'data'/'golden_curated_44.jsonl').exists()
print('Project:', project)
print('Curated labels:', project/'q3'/'data'/'golden_curated_44.jsonl')


Saving HW5_Q3_Curated_Revision_20261118.zip to HW5_Q3_Curated_Revision_20261118.zip
Project: /content/HW5_Q3_Recheck/HW5_Official_Integrated_Code
Curated labels: /content/HW5_Q3_Recheck/HW5_Official_Integrated_Code/q3/data/golden_curated_44.jsonl


## 2. Ensure the small CPU dependencies are present

No Spark is used for the Q3 recheck. The base requirements for these scripts are NumPy and scikit-learn.

In [2]:
%pip -q install "numpy>=1.24,<3" "scikit-learn>=1.3,<2" pytest


## 3. Verify label-source integrity, then evaluate the reviewed Q3 set

Expected counts: **44 queries (36 answerable + 8 unanswerable)**. The model code is unchanged. Failure types should include **4 retrieval misses** and **2 false abstentions after relevant evidence was retrieved**.

In [3]:
import subprocess,sys,os

env = os.environ.copy()
env['PYTHONPATH'] = str(project)
def run_command(*args):
    result = subprocess.run([sys.executable,*args],cwd=str(project),env=env,capture_output=True,text=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f'Failed: {args}')

run_command('-m','pytest','-q','tests/test_q3_curated_labels.py')
run_command('-m','q3.run','--label-set','curated')


....                                                                     [100%]
4 passed in 0.02s

{
  "baseline": {
    "n": 44,
    "answerable": 36,
    "unanswerable": 8,
    "retrieval_recall_at3": 0.9166666666666666,
    "citation_precision_answerable": 0.7222222222222222,
    "fact_match_answerable": 0.7222222222222222,
    "strict_source_backed_accuracy": 0.5909090909090909,
    "failure_categories": {
      "wrong_or_missing_citation": 10,
      "unsupported_answer": 8
    }
  },
  "improved": {
    "n": 44,
    "answerable": 36,
    "unanswerable": 8,
    "retrieval_recall_at3": 0.8888888888888888,
    "citation_precision_answerable": 0.8333333333333334,
    "fact_match_answerable": 0.8333333333333334,
    "strict_source_backed_accuracy": 0.8636363636363636,
    "failure_categories": {
      "retrieval_miss_then_abstention": 4,
      "false_abstention_despite_retrieved_evidence": 2
    }
  },
  "cache": {
    "miss": 44,
    "exact": 88
  },
  "failures": 6
}



## 4. (Recommended) Run the exact **original-label** comparison

This command does **not** change any documents or gold labels. It writes the old-label outputs into a separate directory for a transparent comparison.

In [4]:
run_command('-m','q3.run','--label-set','original')


{
  "baseline": {
    "n": 44,
    "answerable": 36,
    "unanswerable": 8,
    "retrieval_recall_at3": 0.9166666666666666,
    "citation_precision_answerable": 0.7222222222222222,
    "fact_match_answerable": 0.7222222222222222,
    "strict_source_backed_accuracy": 0.5909090909090909,
    "failure_categories": {
      "wrong_or_missing_citation": 10,
      "unsupported_answer": 8
    }
  },
  "improved": {
    "n": 44,
    "answerable": 36,
    "unanswerable": 8,
    "retrieval_recall_at3": 0.8888888888888888,
    "citation_precision_answerable": 0.8333333333333334,
    "fact_match_answerable": 0.8055555555555556,
    "strict_source_backed_accuracy": 0.8409090909090909,
    "failure_categories": {
      "answer_fact_missing_or_unverified": 1,
      "retrieval_miss_then_abstention": 4,
      "false_abstention_despite_retrieved_evidence": 2
    }
  },
  "cache": {
    "miss": 44,
    "exact": 88
  },
  "failures": 7
}



## 5. Inspect the observed metrics and the six remaining failures

In [5]:
import json
cur = json.loads((project/'q3'/'outputs'/'run_info.json').read_text())
old = json.loads((project/'q3'/'outputs_original_label_baseline'/'run_info.json').read_text())
print('Q3 corpus:',cur['docs'],'docs;',cur['chunks'],'chunks')
print('Reviewed query count:',cur['labelled_queries'])
print('Improved Recall@3:',round(cur['improved']['retrieval_recall_at3']*100,2),'%')
print('Original strict accuracy:',round(old['improved']['strict_source_backed_accuracy']*100,2),'%')
print('Curated strict accuracy:',round(cur['improved']['strict_source_backed_accuracy']*100,2),'%')
print('Failure categories:',cur['improved']['failure_categories'])
assert cur['labelled_queries']==44
assert len(json.loads((project/'q3'/'outputs'/'q3_failures_to_analyze.json').read_text()))==6


Q3 corpus: 779 docs; 943 chunks
Reviewed query count: 44
Improved Recall@3: 88.89 %
Original strict accuracy: 84.09 %
Curated strict accuracy: 86.36 %
Failure categories: {'retrieval_miss_then_abstention': 4, 'false_abstention_despite_retrieved_evidence': 2}


## 6. Download Q3 experiment outputs only

This ZIP contains reviewed labels, audit log and Q3 output metrics. The full working-code package is separate; do not submit the experiment-only ZIP as the final coursework ZIP.

In [6]:
from google.colab import files
from pathlib import Path
import zipfile
output = Path('/content/HW5_Q3_Curated_Evaluation_Results.zip')
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in ['q3/data/golden_curated_44.jsonl','q3/label_change_log.csv','q3/label_manifest.json']:
        p=project/rel
        if p.exists():z.write(p,rel)
    for sub in ['q3/outputs','q3/outputs_original_label_baseline']:
        for p in (project/sub).rglob('*'):
            if p.is_file():z.write(p,p.relative_to(project))
print('Saved:',output,'bytes:',output.stat().st_size)
files.download(str(output))


Saved: /content/HW5_Q3_Curated_Evaluation_Results.zip bytes: 25124


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reporting note

Report the **curated** score as `38/44 (86.36%)` on a **user-confirmed, AI-assisted 44-query development set**, and note that the original-label score was `37/44 (84.09%)` with the unchanged model. For the final EECS E6893 submission, keep Q1/Q2 results separate, use your actual Colab hardware when reporting Colab latency, and follow the course AI-assistance disclosure policy.